In [1]:
import sys

import polars as pl
import torch

from modeling_module.data_loader.MultiPartDataModule import MultiPartDataModule
from modeling_module.models.PatchMixer.common.configs import PatchMixerConfig
from modeling_module.utils.checkpoint import save_model_dict, load_model_dict

'''
pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
https://developer.nvidia.com/cuda-12-8-0-download-archive
'''

MAC_DIR = '/Users/igwanhyeong/PycharmProjects/data_research/raw_data/'
WINDOW_DIR = 'C:/Users/USER/PycharmProjects/research/raw_data/'

if sys.platform == 'win32':
    DIR = WINDOW_DIR
    print(torch.cuda.is_available())
    print(torch.cuda.device_count())
    print(torch.version.cuda)
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    print(torch.__version__)
else:
    DIR = MAC_DIR

save_dir = DIR + 'fit/20251106_running'


True
1
12.8
2.9.0.dev20250716+cu128
NVIDIA GeForce RTX 5080
2.9.0.dev20250716+cu128


In [2]:
target_dyn_demand_weekly = pl.read_parquet(DIR + 'target_dyn_demand_weekly.parquet').sort(['oper_part_no', 'demand_dt'])
target_dyn_demand_weekly = (target_dyn_demand_weekly.group_by('oper_part_no', maintain_order = True).map_groups(lambda g: g.with_columns(pl.arange(1, len(g) + 1).alias('seq'))))

filtered_target = target_dyn_demand_weekly.group_by('oper_part_no').agg(pl.col('seq').max().alias('seq_max')).filter(pl.col('seq_max') > 260).select('oper_part_no')

target_dyn_demand_weekly = target_dyn_demand_weekly.join(filtered_target, on = 'oper_part_no', how = 'right').select(['oper_part_no', 'demand_dt', 'demand_qty'])
target_dyn_demand_weekly

oper_part_no,demand_dt,demand_qty
str,i64,f64
"""82EV-0508-1""",201801,39.0
"""82EV-0508-1""",201802,156.0
"""82EV-0508-1""",201803,17.0
"""82EV-0508-1""",201804,8.0
"""82EV-0508-1""",201805,3.0
…,…,…
"""E7312-32431""",202701,280.0
"""E7312-32431""",202702,407.0
"""E7312-32431""",202703,45.0


In [3]:
plan_yyyymm = 201811
lookback = 52
horizon = 27
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

data_module = MultiPartDataModule(
    target_dyn_demand_weekly.sort(['oper_part_no', 'demand_dt']),
    lookback = lookback,
    horizon = horizon,
    batch_size = 64,
    val_ratio = 0.2,
    is_running = True
)

train_loader = data_module.get_train_loader()
val_loader = data_module.get_val_loader()

In [ ]:
from modeling_module.training.model_trainers.total_train import run_total_train_weekly

mode_dict = run_total_train_weekly(
    train_loader,
    val_loader,
    lookback = lookback,
    horizon = horizon,
)

PatchMixer Base (Weekly)
[train_patchmixer] Effective TrainingConfig:
{
  "device": "cuda",
  "lookback": 52,
  "horizon": 27,
  "epochs": 3,
  "lr": 0.0003,
  "weight_decay": 0.0001,
  "t_max": 40,
  "patience": 8,
  "max_grad_norm": 30.0,
  "amp_device": "cuda",
  "loss_mode": "quantile",
  "point_loss": "mse",
  "huber_delta": 5.0,
  "q_star": 0.5,
  "use_cost_q_star": false,
  "Cu": 1.0,
  "Co": 1.0,
  "quantiles": [
    0.1,
    0.5,
    0.9
  ],
  "use_intermittent": true,
  "alpha_zero": 1.2,
  "alpha_pos": 1.0,
  "gamma_run": 0.6,
  "cap": null,
  "use_horizon_decay": false,
  "tau_h": 0.85,
  "val_use_weights": false,
  "spike_loss": {
    "enabled": true,
    "strategy": "mix",
    "huber_delta": 0.9,
    "asym_up_weight": 1.0,
    "asym_down_weight": 8.0,
    "mad_k": 1.8,
    "w_spike": 24.0,
    "w_norm": 1.0,
    "alpha_huber": 0.6,
    "beta_asym": 0.4,
    "mix_with_baseline": false,
    "gamma_baseline": 0.0
  }
}
[CommonTrainer] TrainingConfig (final)
{
  "device": "c

C:\Users\USER\python\py312\Lib\site-packages\torch\nn\modules\conv.py:366: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1028.)
  return F.conv1d(


Epoch 1/3 | LR 0.000300 | Train 31.642783 | Val 27.134906
Epoch 2/3 | LR 0.000298 | Train 28.576519 | Val 27.024130
Epoch 3/3 | LR 0.000296 | Train 27.579373 | Val 27.053565
[EXO-train] model.exo_dim=2  future_exo_cb? True  exo_is_normalized=True
PatchMixer Quantile (Weekly)
[train_patchmixer] Effective TrainingConfig:
{
  "device": "cuda",
  "lookback": 52,
  "horizon": 27,
  "epochs": 3,
  "lr": 0.0003,
  "weight_decay": 0.0001,
  "t_max": 40,
  "patience": 8,
  "max_grad_norm": 30.0,
  "amp_device": "cuda",
  "loss_mode": "quantile",
  "point_loss": "mse",
  "huber_delta": 5.0,
  "q_star": 0.5,
  "use_cost_q_star": false,
  "Cu": 1.0,
  "Co": 1.0,
  "quantiles": [
    0.1,
    0.5,
    0.9
  ],
  "use_intermittent": true,
  "alpha_zero": 1.2,
  "alpha_pos": 1.0,
  "gamma_run": 0.6,
  "cap": null,
  "use_horizon_decay": false,
  "tau_h": 0.85,
  "val_use_weights": false,
  "spike_loss": {
    "enabled": true,
    "strategy": "mix",
    "huber_delta": 0.9,
    "asym_up_weight": 1.0,
 

In [ ]:
from modeling_module.models.Titan.common.configs import TitanConfig

pm_base_config = PatchMixerConfig(
        lookback = lookback,
        horizon = horizon,
        device = device,
        loss_mode = 'point',
        point_loss = 'mae'
    )

pm_quantile_config = PatchMixerConfig(
    lookback = lookback,
        horizon = horizon,
    device = device,
    loss_mode = 'quantile',
    quantiles = (0.1, 0.5, 0.9)
)

# ti_config = TitanConfig(
#             lookback=lookback,
#             horizon=horizon,
#             # 아래는 Titans.py에서 사용하는 공통 옵션들(필요 시 설정)
#             input_dim=1,  # 데이터로더 입력 채널 수에 맞춰 조정
#             d_model=256,
#             n_layers=3,
#             n_heads=4,
#             d_ff=512,
#             dropout=0.1,
#             contextual_mem_size=256, persistent_mem_size=64,
#             use_exogenous=True, exo_dim=2,  # 캘린더 sin/cos 자동 주입 조건
#             final_clamp_nonneg=False,
#         )
#
# pt_config = PatchTSTConfigMonthly(
#         device = device,
#         loss_mode = 'auto',
#         quantiles = (0.1, 0.5, 0.9)
#     )

cfg_map = {
    "PatchMixer Base": pm_base_config,
    "PatchMixer Quantile": pm_quantile_config,
    # "Titan Base": ti_config,
    # "Titan LMM": ti_config,
    # "Titan Seq2Seq": ti_config,
    # "PatchTST Base": pt_config,
    # "PatchTST Quantile": pt_config
}

In [ ]:

models_only = {
    name: (pack["model"] if isinstance(pack, dict) and "model" in pack else pack)
    for name, pack in mode_dict.items()
}

# cfg_by_name도 맞춰서 준비(필요하면)
cfg_by_name = {
    name: (pack.get("cfg") if isinstance(pack, dict) else None)
    for name, pack in mode_dict.items()
}

builder_key_by_name = {
  # "PatchMixer Base": "patchmixer_base",
  # "PatchMixer Quantile": "patchmixer_quantile",
  "Titan Base": "titan_base",
  "Titan LMM": "titan_lmm",
  "Titan Seq2Seq": "titan_seq2seq",
  # "PatchTST Base": "patchtst_base",
  # "PatchTST Quantile": "patchtst_quantile",
}
save_index = save_model_dict(
    models_only,
    save_dir,
    cfg_by_name=cfg_by_name,                 # None 가능
    builder_key_by_name=builder_key_by_name  # 기존 그대로
)

In [ ]:

# Load
from modeling_module.models.model_builder import (
    build_titan_base, build_titan_lmm, build_titan_seq2seq, build_patch_mixer_base, build_patch_mixer_quantile,
)

builders = {
    "patchmixer_base": lambda cfg: build_patch_mixer_base(cfg or PatchMixerConfig()),
    "patchmixer_quantile": lambda cfg: build_patch_mixer_quantile(cfg or PatchMixerConfig()),
    # "titan_base": lambda cfg: build_titan_base(ti_config or TitanConfig()),
    # "titan_lmm": lambda cfg: build_titan_lmm(ti_config or TitanConfig()),
    # "titan_seq2seq": lambda cfg: build_titan_seq2seq(ti_config or TitanConfig()),
    # "patchtst_base": lambda cfg: build_patchTST_base(cfg or PatchTSTConfigMonthly()),
    # "patchtst_quantile": lambda cfg: build_patchTST_quantile(cfg or PatchTSTConfigMonthly()),
}
loaded = load_model_dict(save_dir, builders, device = device)

In [ ]:
# # 배치 꺼내기
# xb, yb = next(iter(val_loader))[:2]
# xb = xb.to(device)
# Hm = ti_config.horizon
#
# # exo 구성 (캘린더 52주기 가정)
# from modeling_module.training import forecaster as fo
# exo = fo.make_calendar_exo(0, Hm, period=52, device=device).unsqueeze(0).expand(xb.size(0), -1, -1)
#
# # 한 번의 forward
# model = loaded["titan_base"].to(device).eval()  # 또는 titan_seq2seq / titan_lmm
# with torch.no_grad():
#     yhat = model(xb, future_exo=exo)  # [B,H]
#
# print("yhat shape:", tuple(yhat.shape))
# print("min/max:", float(yhat.min()), float(yhat.max()))
# print("has_all_zero:", bool((yhat.abs() < 1e-8).all().item()))

model = loaded["titan_base"].to(device).eval()   # 또는 "titan_seq2seq"/"titan_lmm"

# 배치
xb, yb = next(iter(val_loader))[:2]
xb = xb.to(device)
Hm = getattr(model, "horizon", 27)

# exo 만들기 (sin, cos)
from modeling_module.training import forecaster as fo
exo = fo.make_calendar_exo(0, Hm, period=52, device=device).unsqueeze(0).expand(xb.size(0), -1, -1)

# 1) 파라미터 노름 점검
def pnorm(m, name):
    p = dict(m.named_parameters())
    s = {k: float(v.detach().abs().sum()) for k,v in p.items()}
    print(name, "param S1:", s)

pnorm(model, "MODEL")
print("proj.weight sum:", float(model.proj.weight.detach().abs().sum()) if hasattr(model,"proj") else "no-proj")
print("has query/pos:", hasattr(model.decoder, "query_embed"), hasattr(model.decoder, "pos_embed"))
if hasattr(model.decoder, "query_embed"):
    print("query_embed abs sum:", float(model.decoder.query_embed.detach().abs().sum()))
if hasattr(model.decoder, "pos_embed"):
    print("pos_embed abs sum:", float(model.decoder.pos_embed.detach().abs().sum()))

# 2) 중간 출력 확인용 훅
enc_out_stats = {}
dec_out_stats = {}

def enc_hook(mod, inp, out):
    enc_out_stats["mean"] = float(out.detach().mean())
    enc_out_stats["amax"] = float(out.detach().abs().max())
def dec_hook(mod, inp, out):
    dec_out_stats["mean"] = float(out.detach().mean())
    dec_out_stats["amax"] = float(out.detach().abs().max())

h1 = model.encoder.register_forward_hook(enc_hook)
h2 = model.decoder.register_forward_hook(dec_hook)

with torch.no_grad():
    yhat = model(xb, future_exo=exo)  # [B,H] 또는 [B,H,1]

h1.remove(); h2.remove()

print("enc_out:", enc_out_stats)
print("dec_out:", dec_out_stats)
print("yhat shape:", tuple(yhat.shape), "min/max:", float(yhat.min()), float(yhat.max()))

In [ ]:
%load_ext autoreload
%autoreload 2

import importlib, modeling_module.utils.plot_utils as pu
import modeling_module.training.forecaster as fo
importlib.reload(pu)
importlib.reload(fo)

def my_exo_cb(start_idx: int, Hm: int, device="cuda" if torch.cuda.is_available() else "cpu"):
    # exo_dim = 2 (sin, cos)
    return fo.make_calendar_exo(start_idx, Hm, period=52, device=device)

pu.plot_27w(
    models=loaded,           # {"PatchMixer": pm_model, "Titan": ti_model, ...}
    loader=val_loader,       # (xb, yb[, part_ids])
    device="cuda" if torch.cuda.is_available() else "cpu",
    mode="val",              # ← 검증 모드
    max_plots=5,
    out_dir=None,
    show=True,
    future_exo_cb=my_exo_cb
)